# Day 7 — Classes & Objects
### Python for Data Science · Module 1 · Topic 1.7

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | The idea: why classes, and the blueprint metaphor | 20 min |
| 2 | `__init__` and `self` | 25 min |
| 3 | Methods | 25 min |
| 4 | Class attributes, `__str__`, and when *not* to use a class | 15 min |
| 5 | Mini build: a `BankAccount` class | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **You have been using objects since Day 1.** Every `.append()` on a list, every `.get()`
> on a dictionary, every `.strip()` on a string was a method call on an object. Today you
> find out what was behind the dot — and start writing your own.

---
## 0. Recap of Day 6

In [ ]:
nums = [3, -1, 4]

print([n for n in nums if n > 0])                 # filter - can be shorter
print(["pos" if n > 0 else "neg" for n in nums])  # choose - same length
print({n for n in nums})                          # a set, no colon
print(type((n for n in nums)).__name__)           # a generator, not a tuple

That closes the data-structures half of Module 1. From today you build your own types.

---
# 1. The idea

## 1.1 Why not just use a dictionary?

In [ ]:
# YESTERDAY'S APPROACH - data in a dict, behaviour in a separate function
ravi = {"name": "Ravi", "marks": [88, 71]}
sara = {"name": "Sara", "marks": [91, 84]}

def average(student):
    return sum(student["marks"]) / len(student["marks"])

print(average(ravi))

# But nothing stops this:
ravi["nmae"] = "typo"        # silently accepted
print(ravi.keys())           # now there is a junk key nobody will notice

In [ ]:
# TODAY'S APPROACH - the data and what you do with it travel together
class Student:
    def __init__(self, name, marks):
        self.name  = name
        self.marks = marks

    def average(self):
        return sum(self.marks) / len(self.marks)


ravi = Student("Ravi", [88, 71])
print(ravi.average())
print(ravi.name)

**Four things a class buys you:**

- **One definition** — every `Student` is built the same way
- **Data + behaviour together** — `average()` lives with the marks it needs
- **Clear errors** — miss an argument and it fails immediately, not later
- **Readable** — `ravi.average()` says what it is

## 1.2 A class is a blueprint; an object is one thing built from it

The class defines what every `Student` will have. It creates nothing on its own —
Day 3's *"defining is not running"*, again.

In [ ]:
class Student:
    def __init__(self, name):
        self.name = name

# Running the cell above created NO students. Now we build three:
ravi = Student("Ravi")
sara = Student("Sara")
amit = Student("Amit")

print(ravi.name, sara.name, amit.name)
print(type(ravi))
print(isinstance(ravi, Student))

Creating an object looks like calling a function — because it is. The class name with
brackets after it builds a new object and hands it back. The arguments you pass go
straight into `__init__`.

---
# 2. `__init__` and `self`

```python
class Student:
    def __init__(self, name, marks):
        self.name  = name
        self.marks = marks
```

| | |
|---|---|
| `__init__` | runs automatically when you create an object |
| `self` | the object being built, handed in by Python |
| `self.name` | an attribute stored **on** that object |
| `name` | just a parameter — gone when `__init__` ends |

**What happens on creation:**

1. `ravi = Student("Ravi", [88])`
2. Python makes a blank object
3. Python calls `__init__(that object, "Ravi", [88])`
4. The filled object comes back to you

## 2.1 `self` is not magic — it is just the first parameter

You never pass `self` yourself. Python puts the object before the dot into that first slot.

In [ ]:
class Student:
    def __init__(self, name, marks):
        self.name  = name
        self.marks = marks

    def average(self):
        return sum(self.marks) / len(self.marks)


ravi = Student("Ravi", [88, 71])

print(ravi.average())            # what you write
print(Student.average(ravi))     # what Python actually runs - identical

The name `self` is only a **convention** — Python would accept any name — but every Python
programmer expects it, so always use it.

## 2.2 Instance attributes — each object's own data

In [ ]:
ravi = Student("Ravi", [88, 71])
sara = Student("Sara", [91, 84])

print(ravi.name, sara.name)

ravi.name = "Ravi Kumar"      # change just this one object
print(ravi.name, sara.name)   # sara is unaffected

### ⚠️ Three mistakes with `self`

In [ ]:
# MISTAKE 1: forgetting self in the parameter list
class Broken1:
    def __init__(name):        # no self
        self.name = name

try:
    Broken1("Ravi")
except TypeError as e:
    print("TypeError:", e)
    print("  -> Python DID pass the object, so there is one more argument than expected.")

In [ ]:
# MISTAKE 2: assigning to a bare name instead of self.name
class Broken2:
    def __init__(self, name):
        name = name            # does nothing useful

b = Broken2("Ravi")
try:
    print(b.name)
except AttributeError as e:
    print("AttributeError:", e)

In [ ]:
# MISTAKE 3: one underscore instead of two
class Broken3:
    def _init_(self, name):    # NOT the initialiser
        self.name = name

b = Broken3()                  # no error - it just never ran
print("created, but:", hasattr(b, "name"))

### Attributes can be added later — but usually shouldn't be

In [ ]:
ravi = Student("Ravi", [88])
ravi.city = "Pune"            # not in __init__ at all
print(ravi.city)

sara = Student("Sara", [91])
try:
    print(sara.city)
except AttributeError as e:
    print("AttributeError:", e)

# Python allows this, but now some Students have a city and others do not,
# and any code reading .city will crash unpredictably.
# Put everything an object needs in __init__.

---
# 3. Methods — functions that live inside a class

In [13]:
class Student:
    def __init__(self,name, marks):
        self.name  = name
        self.marks = marks

    def average(self):                    # READS its own data
        return sum(self.marks) / len(self.marks)

    def add_mark(self, mark):             # CHANGES its own data
        self.marks.append(mark)
        return self.marks
    
    def rename(self,new_name):
        self.name = new_name

    def report(self):                     # calls another method
        return f"{self.name}: {self.average():.1f}"


ravi = Student("Ravi", [1, 2])
print(ravi.average())        # 79.5

print(ravi.add_mark(3))
print(ravi.average())        # 83.0
print(ravi.report())

sara = Student("Sara",[20,40])
print(sara.average())
print(sara.report())

sara.rename("Sara Singh")

print(sara.report())

print(ravi)

1.5
[1, 2, 3]
2.0
Ravi: 2.0
30.0
Sara: 30.0
Sara Singh: 30.0


A method is a normal function with one extra rule. Everything from Day 3 still applies:
default arguments, keyword arguments, `*args`, early return, docstrings.

The only difference is that the first parameter is always `self`, and through it the method
can reach the object's own data.

In [ ]:
# Inside a method, always self.x - a bare x is a NameError
class Broken:
    def __init__(self, marks):
        self.marks = marks

    def total(self):
        return sum(marks)      # forgot self.

b = Broken([1, 2, 3])
try:
    b.total()
except NameError as e:
    print("NameError:", e)

## 3.1 Reading state versus changing state

In [ ]:
class Account:
    def __init__(self, balance=0):
        self.balance = balance

    # ASKS - returns a value, changes nothing
    def is_overdrawn(self):
        return self.balance < 0

    # DOES - changes self, returns nothing
    def deposit(self, amount):
        self.balance += amount


acc = Account(100)
print(acc.is_overdrawn())     # False - just asking

acc.deposit(50)               # doing - no value expected back
print(acc.balance)

### ⚠️ Never assign the result of a method that changes state

In [ ]:
acc = Account(100)

acc = acc.deposit(50)         # deposit returns None...
print(acc)                    # ...so acc is now None!

In [ ]:
# Correct
acc = Account(100)
acc.deposit(50)
print(acc.balance)

# This is exactly Day 4's  nums = nums.append(4)  mistake in different clothes.
# Keep the two kinds separate: a method should either answer a question
# or change something, not both.

---
# 4. Beyond the basics

## 4.1 Class attributes — shared by every object

In [ ]:
class Student:
    school = "Boston University"      # CLASS attribute - one, shared

    def __init__(self, name):
        self.name = name              # INSTANCE attribute - one per object


ravi = Student("Ravi")
sara = Student("Sara")

print(ravi.school, "|", sara.school)     # same for all
print(ravi.name,   "|", sara.name)       # different

- **Instance attribute** — anything that differs per object: name, marks, balance
- **Class attribute** — anything genuinely shared: a constant, a default, a running count

### ⚠️ Never make a class attribute a list or dict

This is the **fifth** appearance of aliasing.

In [ ]:
class Broken:
    marks = []                # ONE list, shared by ALL objects

    def __init__(self, name):
        self.name = name

    def add(self, m):
        self.marks.append(m)


ravi = Broken("Ravi")
sara = Broken("Sara")

ravi.add(88)
print("sara.marks:", sara.marks)     # [88] - sara got Ravi's mark!
print("same object?", ravi.marks is sara.marks)

In [ ]:
# THE FIX: mutable data belongs in __init__, so each object gets its own
class Fixed:
    def __init__(self, name):
        self.name  = name
        self.marks = []       # a NEW list per object

    def add(self, m):
        self.marks.append(m)


ravi = Fixed("Ravi")
sara = Fixed("Sara")
ravi.add(88)

print("ravi:", ravi.marks, "| sara:", sara.marks)
print("same object?", ravi.marks is sara.marks)

> ### One idea, five appearances
> Day 1's `c = a` · Day 3's mutable default argument · Day 3's mutable argument ·
> Day 4's `b = a` · today's mutable class attribute.
>
> All the same sentence: **two names, one object.**

## 4.2 Making objects printable — `__str__`

In [16]:
class Student:
    def __init__(self, name):
        self.name = name

ravi = Student("Ravi")
print(ravi)      # technically correct, completely useless

In [22]:
class Student:
    def __init__(self, name,marks):
        self.name = name
        self.marks = marks

    def __str__(self):
        return f"Student name :{self.name}, {self.name} Marks : {self.marks}"


ravi = Student("Ravi",[1,2,3])
print(ravi)          # print() uses __str__
# print(str(ravi))

Student name :Ravi, Ravi Marks : [1, 2, 3]


### The dunder methods you will meet next

| Method | What it does |
|---|---|
| `__init__` | build the object |
| `__str__` | what `print()` shows |
| `__repr__` | what the notebook shows when you evaluate the object |
| `__len__` | makes `len(obj)` work |
| `__eq__` | makes `obj1 == obj2` work |

*"Dunder"* = double underscore. Python calls them for you.

In [ ]:
class Playlist:
    def __init__(self, songs):
        self.songs = songs

    def __len__(self):
        return len(self.songs)

    def __str__(self):
        return f"Playlist({len(self)} songs)"


p = Playlist(["a", "b", "c"])
print(len(p))     # __len__ makes this work
print(p)          # __str__

## 4.3 When a class is NOT the answer

| Situation | Better tool |
|---|---|
| Just holding data? | a dict is simpler |
| No data at all, only steps? | a function is enough |
| Only ever one of them? | you do not need a blueprint |
| A class with one method? | that method is just a function |

> A class earns its place when **several objects each carry their own data** *and*
> **there is behaviour that belongs with that data**. Both halves matter — data alone is a
> dictionary, behaviour alone is a function.

---
# 5. Putting it together — a `BankAccount` class

In [ ]:
class BankAccount:
    """A single account with a balance and a transaction history."""

    bank = "BU Savings"                       # class attribute - shared

    def __init__(self, owner, balance=0):     # default argument - Day 3
        self.owner   = owner
        self.balance = balance
        self.history = []                     # own list per object

    def deposit(self, amount):                # CHANGES state
        if amount <= 0:                       # early return - Day 3
            return "amount must be positive"
        self.balance += amount
        self.history.append(f"+{amount}")

    def withdraw(self, amount):
        if amount > self.balance:
            return "insufficient funds"
        self.balance -= amount
        self.history.append(f"-{amount}")

    def statement(self):                      # ASKS, returns
        return f"{self.owner}: {self.balance} ({len(self.history)} txns)"

    def __str__(self):
        return f"BankAccount({self.owner}, {self.balance})"

In [ ]:
a = BankAccount("Ravi", 100)
b = BankAccount("Sara")           # balance defaults to 0

a.deposit(50)
a.withdraw(30)

print(a.statement())
print(b.statement())

print("guards:", a.deposit(-5), "|", b.withdraw(999))
print("print():", a)
print("shared bank:", a.bank, b.bank)
print("independent histories:", a.history, b.history)

Every idea from today, plus four from Day 3:

- **`__init__`** with `owner` and a defaulted `balance`
- **Default argument** — `balance=0`
- **Own list per object** — `history` in `__init__`, not the class body
- **Class attribute** — `bank` is shared and constant
- **Changes state** — `deposit` / `withdraw` return nothing on success
- **Asks** — `statement` returns a value
- **Early return** — the guards run first
- **`__str__`** — `print(acc)` reads properly

---
# 6. Recap — the twelve things to remember

1. A class is a blueprint; an object is one thing built from it.
2. `ClassName(...)` creates an object and runs `__init__`.
3. `__init__` needs **two** underscores each side, and `self` first.
4. `self` is the object itself, passed in by Python for you.
5. `obj.method()` is really `Class.method(obj)`.
6. `self.name = name` stores data **on** the object.
7. Inside a method, always `self.x` — a bare `x` is a `NameError`.
8. Methods either **ASK** (return) or **DO** (change `self`).
9. Never assign the result of a method that changes state.
10. Class attributes are shared; never make one a list or dict.
11. `__str__` decides what `print(obj)` shows.
12. Data only? Use a dict. Steps only? Use a function.

---

### 📝 Now open **`Day7_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Write a `Book` class with title, author, pages and a `summary()` method.
- Add `__str__` to it.
- Rewrite Day 5's gradebook as a `Gradebook` class.

### Next class — Topic 1.8: Inheritance & Polymorphism
Parent and child classes, method overriding, `super()`, and the basics of multiple inheritance.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*